#Writing the StockOption base class

In [ ]:
import math
class stockoption(object):
  def __init__(self, s0, k, r , T, N, params):
    self.s0 = s0
    self.k = k
    self.r = r
    self.T = T
    self.N = max(1, N)
    self.STs = None # Declare the stock prices tree

    """ Optional parameters used by derived classes """
    self.pu = params.get("pu", 0) # Probability of up state
    self.pd = params.get("pd", 0) # Probability of down state
    self.div = params.get("div", 0) # Dividend yield
    self.sigma = params.get("sigma", 0) # Volatility
    self.is_call = params.get("is_call", True) # Call or put
    self.is_european = params.get("is_eu", True) # Eu or Am

    """ Computed values """
    self.dt = T/float(N) # Single time step, in years
    self.df = math.exp(-(r-self.div) * self.dt) # Discount factor


In [ ]:
option1 = stockoption( s0=100, k=110, r=0.05, T=1, N=2, params={"pu": 0.5, "pd": 0.5, "div": 0})

In [ ]:
print(option1.dt)
print(option1.df)

0.5
0.9753099120283326


In [ ]:
print("Stock Price:", option1.s0)
print("Strike Price:", option1.k)
print("Risk-free Rate:", option1.r)
print("Time to Maturity:", option1.T)
print("Number of Steps:", option1.N)
print("Time Step:", option1.dt)
print("Discount Factor:", option1.df)

Stock Price: 100
Strike Price: 110
Risk-free Rate: 0.05
Time to Maturity: 1
Number of Steps: 2
Time Step: 0.5
Discount Factor: 0.9753099120283326


In [ ]:
# Rounded to two decimal places   :.2f
print("Discount Factor:", f"{option1.df:.2f}")

Discount Factor: 0.98


# A class for European options using a binomial tree

In [ ]:
#Price a European option by the binomial tree model
#from stockoption import stockoption
import math
import numpy as np
class BinomialEuropeanOption(stockoption):
  def _setup_parameters(self):
    self.M = self.N+1 # Number of terminal nodes of tree
    self.u = 1+self.pu # Expected value in the up state
    self.d = 1-self.pd # Expected value in the down state
    self.qu = (math.exp((self.r-self.div)*self.dt) - self.d) / (self.u-self.d)
    self.qd = 1-self.qu

_IncompleteInputError: incomplete input (3608974717.py, line 6)

In [ ]:
    import math
    import numpy as np
    from decimal import Decimal

    """
    Price a European option by the binomial tree model
    """
    class BinomialEuropeanOption(StockOption):

        def setup_parameters(self):
            # Required calculations for the model
            self.M = self.N+1  # Number of terminal nodes of tree
            self.u = 1+self.pu  # Expected value in the up state
            self.d = 1-self.pd  # Expected value in the down state
            self.qu = (math.exp(
                (self.r-self.div)*self.dt)-self.d)/(self.u-self.d)
            self.qd = 1-self.qu

        def init_stock_price_tree(self):
            # Initialize terminal price nodes to zeros
            self.STs = np.zeros(self.M)

            # Calculate expected stock prices for each node
            for i in range(self.M):
                self.STs[i] = self.S0 * \
                    (self.u**(self.N-i)) * (self.d**i)

        def init_payoffs_tree(self):
            """
            Returns the payoffs when the option
            expires at terminal nodes
            """
            if self.is_call:
                return np.maximum(0, self.STs-self.K)
            else:
                return np.maximum(0, self.K-self.STs)

        def traverse_tree(self, payoffs):
            """
            Starting from the time the option expires, traverse
            backwards and calculate discounted payoffs at each node
            """
            for i in range(self.N):
                payoffs = (payoffs[:-1]*self.qu +
                           payoffs[1:]*self.qd)*self.df

            return payoffs

        def begin_tree_traversal(self):
            payoffs = self.init_payoffs_tree()
            return self.traverse_tree(payoffs)

        def price(self):
            """ Entry point of the pricing implementation """
            self.setup_parameters()
            self.init_stock_price_tree()
            payoffs = self.begin_tree_traversal()

            # Option value converges to first node
            return payoffs[0]

In [ ]:
eu_option = BinomialEuropeanOption(50, 52, r=0.05, T=2, N=2, pu=0.2, pd=0.2, is_put=True)

In [ ]:
print('European put option price is:', eu_option.price())

European put option price is: 4.1926542806038585
